<cell_type>markdown</cell_type># Step 5: Before vs After 종합 비교

## 학습 목표
이 노트북을 완료하면 다음을 이해할 수 있습니다:
- **Domain Shift 문제**와 Fine-tuning의 효과
- 각 **Fine-tuning 기법의 장단점**
- **적합한 기법 선택 기준**

## 비교 관점

### 1. Before vs After (Domain Adaptation)
```
Before (FF++ Pretrained)     After (Fine-tuned)
       서양인 위주        →      한국인 특화
        ~70%                     ~90%
```

### 2. Fine-tuning 기법 비교
| 관점 | Full | Freeze | LoRA |
|------|------|--------|------|
| 성능 | ⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐ |
| 속도 | ⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| 메모리 | ⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| 과적합 위험 | 높음 | 낮음 | 낮음 |

## 이 노트북에서 확인할 것
1. Before vs 각 After 기법 성능 향상폭
2. 기법별 효율성 (파라미터 수 대비 성능)
3. 우리 태스크에 가장 적합한 기법

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pathlib import Path

# ============================================
# 프로젝트 경로 자동 설정
# ============================================
home_dir = Path.home()
PROJECT_ROOT = home_dir / 'deepfake-detection-sagemaker'
notebook_dir = PROJECT_ROOT / '5_comparison'
os.chdir(notebook_dir)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Current Dir: {os.getcwd()}")

# ============================================
# 결과 파일 로드
# ============================================

# Before 결과
before_path = PROJECT_ROOT / '2_before_evaluation' / 'before_results.json'
with open(before_path, 'r') as f:
    before = json.load(f)

# After 결과 (다중 기법)
after_path = PROJECT_ROOT / '4_after_evaluation' / 'after_results.json'
with open(after_path, 'r') as f:
    after_data = json.load(f)

# 다중 기법 결과 처리
if 'results' in after_data:
    # 새 형식 (다중 기법)
    after_methods = after_data['methods']
    after_results = after_data['results']
else:
    # 이전 형식 (단일 모델)
    after_methods = ['full']
    after_results = {'full': after_data}

print(f"Before 결과: {before_path}")
print(f"After 결과: {after_path}")
print(f"Fine-tuning 기법: {after_methods}")
print("\n✅ 결과 로드 완료!")

In [ ]:
# ============================================
# 📊 종합 비교 테이블
# ============================================

print("=" * 80)
print("          📊 Fine-tuning 효과 종합 비교 (한국인 딥페이크 테스트)")
print("=" * 80)

# 헤더 출력
header = f"{'모델':<20} {'Accuracy':>12} {'Precision':>12} {'Recall':>12} {'F1':>12} {'개선':>10}"
print(header)
print("-" * 80)

# Before 결과
print(f"{'Before (FF++)':<20} {before['accuracy']*100:>11.1f}% {before['precision']*100:>11.1f}% {before['recall']*100:>11.1f}% {before['f1_score']*100:>11.1f}% {'-':>10}")

print("-" * 80)

# After 결과 (각 기법별)
for method in after_methods:
    result = after_results[method]
    improvement = result['accuracy']*100 - before['accuracy']*100
    sign = '+' if improvement > 0 else ''
    print(f"After ({method.upper():<6}){'':<6} {result['accuracy']*100:>11.1f}% {result['precision']*100:>11.1f}% {result['recall']*100:>11.1f}% {result['f1_score']*100:>11.1f}% {sign}{improvement:>8.1f}%p")

print("=" * 80)

# 최고 성능 기법
best_method = max(after_methods, key=lambda m: after_results[m]['accuracy'])
best_improvement = after_results[best_method]['accuracy']*100 - before['accuracy']*100
print(f"\n🏆 최고 성능: {best_method.upper()} (+{best_improvement:.1f}%p 향상)")

In [ ]:
# ============================================
# 📊 시각화: 기법별 성능 비교 차트
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ['accuracy', 'precision', 'recall', 'f1_score']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1 Score']

# --- 차트 1: Accuracy 비교 (막대 그래프) ---
ax1 = axes[0]
models = ['Before'] + [f'After\n({m.upper()})' for m in after_methods]
accuracies = [before['accuracy']*100] + [after_results[m]['accuracy']*100 for m in after_methods]

colors = ['#ff6b6b'] + ['#4ecdc4', '#45b7d1', '#96ceb4'][:len(after_methods)]
bars = ax1.bar(models, accuracies, color=colors)

ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('Fine-tuning 기법별 정확도 비교', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 100)
ax1.axhline(y=70, color='gray', linestyle='--', alpha=0.5, label='Before 기준')
ax1.axhline(y=90, color='green', linestyle='--', alpha=0.5, label='목표')

for bar in bars:
    height = bar.get_height()
    ax1.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=11)

ax1.legend(loc='lower right')

# --- 차트 2: 전체 메트릭 비교 (그룹 막대 그래프) ---
ax2 = axes[1]
x = np.arange(len(metrics))
width = 0.8 / (1 + len(after_methods))

# Before
before_vals = [before[m]*100 for m in metrics]
ax2.bar(x - width * len(after_methods)/2, before_vals, width, label='Before', color='#ff6b6b')

# After (각 기법)
colors_after = ['#4ecdc4', '#45b7d1', '#96ceb4']
for i, method in enumerate(after_methods):
    after_vals = [after_results[method][m]*100 for m in metrics]
    ax2.bar(x - width * len(after_methods)/2 + width * (i+1), after_vals, width, 
            label=f'After ({method.upper()})', color=colors_after[i % len(colors_after)])

ax2.set_ylabel('Score (%)', fontsize=12)
ax2.set_title('전체 메트릭 비교', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(metric_labels)
ax2.set_ylim(0, 100)
ax2.legend(loc='lower right')

plt.tight_layout()
plt.savefig('comparison_chart.png', dpi=150)
plt.show()

print("\n📊 차트가 comparison_chart.png로 저장되었습니다.")

In [ ]:
# ============================================
# 🎯 결론 및 기법 선택 가이드
# ============================================

print("\n" + "=" * 70)
print("                         🎯 종합 결론")
print("=" * 70)

# 최고/최저 성능 기법
best_method = max(after_methods, key=lambda m: after_results[m]['accuracy'])
worst_method = min(after_methods, key=lambda m: after_results[m]['accuracy'])

print(f"""
1️⃣ Domain Shift 문제 해결
   - Before (서양인 위주): {before['accuracy']*100:.0f}%
   - After (한국인 특화): {after_results[best_method]['accuracy']*100:.0f}%
   - ✅ Fine-tuning으로 +{after_results[best_method]['accuracy']*100 - before['accuracy']*100:.0f}%p 향상!

2️⃣ Fine-tuning 기법 비교""")

# 기법별 분석
for method in after_methods:
    acc = after_results[method]['accuracy']*100
    diff = acc - before['accuracy']*100
    if method == 'full':
        analysis = "전체 학습 → 높은 성능, 과적합 주의"
    elif method == 'freeze':
        analysis = "Classifier만 학습 → 빠르고 안정적"
    elif method == 'lora':
        analysis = "Adapter만 학습 → 효율적, LLM에서 인기"
    else:
        analysis = ""
    print(f"   - {method.upper()}: {acc:.1f}% (+{diff:.1f}%p) - {analysis}")

print(f"""
3️⃣ 기법 선택 가이드
   - 성능 최우선 → Full Fine-tuning
   - 빠른 실험/적은 데이터 → Layer Freezing
   - 대규모 모델/효율성 → LoRA
   
   우리 태스크에서는 {best_method.upper()}가 가장 적합!
""")
print("=" * 70)

<cell_type>markdown</cell_type>## 완료!

Before vs After 성능 비교가 완료되었습니다.

### 핵심 학습 내용

1. **Domain Shift 문제**: Pretrained 모델은 학습 데이터와 다른 도메인에서 성능이 저하됨
2. **Fine-tuning 효과**: 타겟 도메인 데이터로 Fine-tuning하면 성능 크게 향상
3. **기법 선택**: 상황에 따라 적합한 Fine-tuning 기법이 다름

### Fine-tuning 기법 요약

| 기법 | 장점 | 단점 | 적합한 경우 |
|------|------|------|------------|
| **Full** | 최고 성능 | 과적합 위험, 느림 | 데이터 충분, 성능 중요 |
| **Freeze** | 빠름, 안정적 | 성능 제한 | 데이터 적음, 빠른 실험 |
| **LoRA** | 효율적, 좋은 성능 | 구현 복잡 | 대규모 모델, LLM |

**➡️ 다음 단계: `6_demo/deploy_and_demo.ipynb`**

최고 성능 모델을 배포하고 실시간 데모를 실행합니다!